In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from sklearn.model_selection import KFold,cross_val_score
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, log_loss

In [4]:
df = pd.read_csv("../Day7/Concrete_Data.csv")
df

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [6]:
X, y = df.drop("Strength",axis=1), df["Strength"]

In [11]:
gdm = XGBRegressor()
kfold = KFold(n_splits=5,shuffle=True,random_state=26)
params = {"learning_rate" : np.linspace(0.01,0.9,10),
          "max_depth" : [None, 2, 3, 4],
          "n_estimators" : [25, 50, 100]
         }
gcv = GridSearchCV(gdm, param_grid=params,cv=kfold)
gcv.fit(X,y)
print("Best params : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best params :  {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score :  0.935401012517735


In [14]:
gdm = XGBRegressor(random_state=26)
kfold = KFold(n_splits=5,shuffle=True,random_state=26)
params = {"learning_rate" : np.linspace(0.01,0.9,10),
          "max_depth" : [None, 2, 3, 4],
          "n_estimators" : [25, 50, 100]
         }
gcv = GridSearchCV(gdm, param_grid=params,cv=kfold) # scoring = r2
gcv.fit(X,y)
print("Best params : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best params :  {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score :  0.935401012517735


In [15]:
test = pd.read_csv("testConcrete.csv")

test

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210
5,150,23,23,114,4,883,638,36
6,480,64,0,292,3,896,776,180
7,393,49,82,132,1,887,830,271
8,284,63,1,138,1,804,725,44
9,206,38,0,103,2,818,719,191


In [21]:
# in gcv by default refit = True [ it automatically retrain model on best paramaters ]
pred = gcv.predict(test)
pred

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

In [23]:
test["Pred_Strength"] = pred
test

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Pred_Strength
0,495,120,0,155,5,866,884,75,68.533485
1,262,129,0,271,2,808,787,174,44.278156
2,201,48,1,215,5,807,839,113,43.303169
3,329,141,0,286,1,881,823,229,36.111191
4,354,14,0,129,2,839,847,210,50.874771
5,150,23,23,114,4,883,638,36,13.594891
6,480,64,0,292,3,896,776,180,46.644585
7,393,49,82,132,1,887,830,271,65.026108
8,284,63,1,138,1,804,725,44,53.041271
9,206,38,0,103,2,818,719,191,56.998302


In [25]:
test = pd.read_csv("testConcrete.csv")
test

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210
5,150,23,23,114,4,883,638,36
6,480,64,0,292,3,896,776,180
7,393,49,82,132,1,887,830,271
8,284,63,1,138,1,804,725,44
9,206,38,0,103,2,818,719,191


In [26]:
# if refit is false
best_model = XGBRegressor(**gcv.best_params_,random_state = 26)
best_model.fit(X,y)
best_model.predict(test)

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)